# 01 · Data Collection

Pull AIESEC MC India exchange application data for **2022-01-01 → 2025-12-31** from the
AIESEC Analytics API, and parse it into a tidy monthly panel.

**Endpoint** — `GET https://analytics.api.aiesec.org/v2/applications/analyze.json`

**Authentication** — a GIS `access_token` passed as a *query parameter*. Without
credentials this notebook automatically falls back to the deterministic offline
reference dataset, so it runs end to end either way.

> ⚠️ Reference-mode output is **simulated**, not real AIESEC data. See the README.

In [1]:
import sys, warnings
from pathlib import Path

# Make the project root importable when running from notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

from src.config import get_settings, configure_logging

settings = get_settings()
configure_logging(settings, "WARNING")
print("Project root:", ROOT)
print("MC:", settings.mc_name, "| target:", settings.target, "| forecast year:", settings.forecast_year)

Project root: /Users/jayedalammansur/Exchange Activity Prediction Platform
MC: AIESEC in India | target: APP | forecast year: 2026


## 1. Check for credentials

Secrets come only from the environment (`.env`, git-ignored) — never from a config file
and never from a notebook cell.

In [2]:
print("Credentials present:", settings.has_credentials)
print("Office id:", settings.office_id)
print("Base URL:", settings.api_base_url)
print("Endpoint:", settings.api["analyze_endpoint"])
print("Window:", settings.collection["start_date"], "->", settings.collection["end_date"])

Credentials present: False
Office id: None
Base URL: https://analytics.api.aiesec.org
Endpoint: /v2/applications/analyze.json
Window: 2022-01-01 -> 2025-12-31


## 2. Collection windows

The pull is split into calendar-month windows. Monthly windowing keeps each payload
small enough to inspect by hand, makes partial failures recoverable (one bad window
doesn't lose a four-year backfill), and matches the forecasting grain.

In [3]:
from src.api.aiesec_api import month_windows

windows = month_windows(settings.collection["start_date"], settings.collection["end_date"])
print(f"{len(windows)} monthly windows")
print("first:", windows[0])
print("last :", windows[-1])

48 monthly windows
first: (datetime.date(2022, 1, 1), datetime.date(2022, 1, 31))
last : (datetime.date(2025, 12, 1), datetime.date(2025, 12, 31))


## 3. Request contract

One request per `(month, programme)`. Note the nested bracket filter syntax the
Analytics API uses.

In [4]:
namespace = settings.api["filter_namespace"]

example = {
    "access_token": "<redacted>",
    "start_date": "2024-05-01",
    "end_date": "2024-05-31",
    f"{namespace}[office_id]": settings.office_id or "<office_id>",
    f"{namespace}[include_child_offices]": "true",
    f"{namespace}[programmes][]": 1,
    "page": 1,
    "per_page": settings.api["per_page"],
}
for key, value in example.items():
    print(f"  {key} = {value}")

  access_token = <redacted>
  start_date = 2024-05-01
  end_date = 2024-05-31
  performance[office_id] = <office_id>
  performance[include_child_offices] = true
  performance[programmes][] = 1
  page = 1
  per_page = 100


## 4. Collect

`collect_exchange_data` handles authentication, retry with exponential backoff,
`Retry-After` on 429, pagination, and per-window failure isolation. If no token is
present we generate the reference dataset instead — same JSON shape, so everything
downstream is identical.

In [5]:
from src.api.aiesec_api import collect_exchange_data, load_raw_responses
from src.api.reference_data import generate_reference_responses

if settings.has_credentials:
    result = collect_exchange_data(settings)
    print(f"{result.windows_succeeded}/{result.windows_attempted} windows OK "
          f"({result.success_rate:.0%})")
    if result.windows_failed:
        print("Failed windows:", result.windows_failed[:3])
else:
    print("No token found - generating the offline reference dataset instead.\n")
    generate_reference_responses(settings)

envelope = load_raw_responses(settings)
envelope["metadata"]

2026-08-02 17:11:21 | WARNING  | src.api.reference_data       | SIMULATED REFERENCE DATA - not real AIESEC operational data. Set AIESEC_ACCESS_TOKEN in .env to collect live data from the Analytics API.


No token found - generating the offline reference dataset instead.



{'source': 'SIMULATED reference dataset (no API credentials available)',
 'warning': 'SIMULATED REFERENCE DATA - not real AIESEC operational data. Set AIESEC_ACCESS_TOKEN in .env to collect live data from the Analytics API.',
 'endpoint': None,
 'mc_name': 'AIESEC in India',
 'office_id': None,
 'collection_start': '2022-01-01',
 'collection_end': '2025-12-31',
 'collected_at': '2026-08-02T17:11:21.533548+05:45',
 'record_count': 144,
 'failed_windows': [],
 'is_reference_data': True,
 'random_seed': 42}

## 5. Inspect one raw payload

The response is an Elasticsearch-style aggregation: nested `buckets[]` arrays carrying
`doc_count` per funnel status and per child office.

In [6]:
import json

record = envelope["records"][0]
print("month:", record["month"], "| product:", record["product"], "| pages:", len(record["pages"]))

page = record["pages"][0]
office = page["analytics"]["offices"]["buckets"][0]
print("\nFirst office bucket (truncated):")
print(json.dumps(office, indent=2)[:900])

month: 2022-01 | product: GV | pages: 1

First office bucket (truncated):
{
  "key": 90000,
  "key_as_string": "AIESEC in Delhi IIT",
  "doc_count": 101,
  "directions": {
    "buckets": [
      {
        "key": "incoming",
        "doc_count": 36,
        "statuses": {
          "buckets": [
            {
              "key": "applied",
              "doc_count": 36
            },
            {
              "key": "achieved",
              "doc_count": 19
            },
            {
              "key": "accepted",
              "doc_count": 13
            },
            {
              "key": "approved",
              "doc_count": 10
            },
            {
              "key": "realized",
              "doc_count": 7
            },
            {
              "key": "finished",
              "doc_count": 7
            },
            {
              "key": "completed",
              "doc_count": 7
            }
          ]
        }
      },
      {
 


## 6. Parse into the tidy panel

`parse_payload` does a **tolerant recursive walk** rather than indexing a fixed path, so
a change in nesting order or an extra grouping level does not break ingestion.

In [7]:
from src.preprocessing.cleaning import build_exchange_dataset

panel = build_exchange_dataset(settings, save=True)
print(panel.shape)
panel.head(8)

2026-08-02 17:11:21 | WARNING  | src.preprocessing.cleaning   | Building dataset from SIMULATED reference data - results are illustrative only


(5760, 16)


,date,year,month,month_name,quarter,entity,product,direction,programme,APP,ACH,ACC,APD,RE,FI,CO
0,2022-01-01,2022,1,Jan,1,AIESEC in Ahmedabad,GTa,incoming,iGTa,3,1,1,1,1,1,1
1,2022-01-01,2022,1,Jan,1,AIESEC in Ahmedabad,GTe,incoming,iGTe,0,0,0,0,0,0,0
2,2022-01-01,2022,1,Jan,1,AIESEC in Ahmedabad,GV,incoming,iGV,5,3,0,0,0,0,0
3,2022-01-01,2022,1,Jan,1,AIESEC in Ahmedabad,GTa,outgoing,oGTa,5,1,1,1,1,1,1
4,2022-01-01,2022,1,Jan,1,AIESEC in Ahmedabad,GTe,outgoing,oGTe,2,0,0,0,0,0,0
5,2022-01-01,2022,1,Jan,1,AIESEC in Ahmedabad,GV,outgoing,oGV,18,7,4,4,4,4,3
6,2022-01-01,2022,1,Jan,1,AIESEC in Bangalore,GTa,incoming,iGTa,4,1,1,1,1,1,1
7,2022-01-01,2022,1,Jan,1,AIESEC in Bangalore,GTe,incoming,iGTe,1,0,0,0,0,0,0


## 7. Validate

Validation is not optional: funnel monotonicity, complete month coverage, no negative
counts, no duplicate cells.

In [8]:
from src.preprocessing.cleaning import validate_dataset

report = validate_dataset(panel, settings)
print("passed:", report.passed)
print("stats :", report.stats)
print("errors:", report.errors or "none")
print("warns :", report.warnings or "none")

passed: True
stats : {'rows': 5760, 'months': 48, 'entities': 20, 'programmes': 6, 'total_APP': 60302}
errors: none
warns : none


In [9]:
# Confirm the funnel invariant directly.
stages = settings.funnel_stages
for earlier, later in zip(stages, stages[1:]):
    violations = int((panel[later] > panel[earlier]).sum())
    print(f"  {earlier} >= {later}: {'OK' if violations == 0 else f'{violations} VIOLATIONS'}")

  APP >= ACH: OK
  ACH >= ACC: OK
  ACC >= APD: OK
  APD >= RE: OK
  RE >= FI: OK
  FI >= CO: OK


## 8. Result

`data/processed/exchange_data.csv` — one row per (month, LC, programme) with the full
`APP → ACH → ACC → APD → RE → FI → CO` funnel.

**Next:** `02_EDA.ipynb`

In [10]:
print("Rows      :", len(panel))
print("Months    :", panel["date"].dt.to_period("M").nunique())
print("Entities  :", panel["entity"].nunique())
print("Programmes:", sorted(panel["programme"].unique()))
print("Total APP :", f"{panel['APP'].sum():,}")
print("\nSaved to:", settings.paths.processed_dataset)

Rows      : 5760
Months    : 48
Entities  : 20
Programmes: ['iGTa', 'iGTe', 'iGV', 'oGTa', 'oGTe', 'oGV']
Total APP : 60,302

Saved to: /Users/jayedalammansur/Exchange Activity Prediction Platform/data/processed/exchange_data.csv
